Simulación Integral: Centro de Distribución y Ventas
Este cuaderno (Notebook) aborda de manera unificada los 5 temas críticos del modelado de sistemas:
1. **Análisis de colas** (Tiempos de espera, servicio).
2. **Procesos de inventario** (Niveles de stock y políticas de revisión).
3. **Representación de sistemas** (Capacidad y utilización).
4. **Construcción de modelos de eventos discretos** (Usando `SimPy`).
5. **Evaluación de métricas operacionales** (Réplicas múltiples e Intervalos de Confianza).

**Escenario de la Vida Real:** Simularemos una tienda de repuestos automotrices.
- Los clientes llegan aleatoriamente haciendo fila (Cola).
- Los vendedores atienden la solicitud (Servicio).
- Si el repuesto está en stock, se vende y el inventario baja (Inventario).
- Cuando el inventario cae a un punto crítico $s$, se solicita al proveedor una cantidad $Q$ que tarda cierto tiempo (Lead Time) en llegar.



In [1]:
!pip install simpy


In [2]:
import simpy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st

# Fijamos una semilla para reproducibilidad inicial (opcional)
np.random.seed(30)
random.seed(30)

# Configuración de gráficos
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (10, 5)


## 1. Definición de Parámetros del Sistema
Aquí definimos las variables críticas abordadas en los fundamentos matemáticos ($\lambda, \mu, c, s, Q$).


In [3]:
# --- Parámetros de Colas (Kendall: M/M/c) ---
LAMBDA = 30.0      # Tasa de llegada: 5 clientes por hora
MU = 10.0          # Tasa de servicio: Cada vendedor atiende a 6 clientes por hora
SERVIDORES = 24    # Cantidad de vendedores ('c')

# --- Parámetros de Inventario (Política Continua s, Q) ---
STOCK_INICIAL = 500
PUNTO_REORDEN_s = 50    # Cuando quedan 20 unidades, se pide más
CANTIDAD_PEDIDO_Q = 400  # Cantidad fija a pedir al proveedor
LEAD_TIME = 2          # El proveedor tarda 24 horas en entregar

# --- Parámetros de Simulación ---
TIEMPO_SIMULACION = 60  # Simular 30 días continuos (en horas)
REPLICAS = 30


## 2. Construcción del Modelo de Eventos Discretos (DES)
Creamos las clases que gobernarán los eventos del sistema usando `SimPy`.
El reloj avanzará de evento a evento (llegada, inicio de atención, fin de atención, llegada de pedido).


In [4]:
class API_MachineLearning:
    def __init__(self, env):
        self.env = env

        # Recurso: Nodos GPU
        self.gpus = simpy.Resource(env, capacity=SERVIDORES)

        # Inventario: Créditos cloud
        self.creditos = simpy.Container(env, init=STOCK_INICIAL, capacity=2000)

        self.punto_reorden = PUNTO_REORDEN_s
        self.cantidad_recarga = CANTIDAD_PEDIDO_Q
        self.lead_time = LEAD_TIME

        self.recarga_pendiente = False

        # Métricas
        self.tiempos_espera = []
        self.predicciones_fallidas = 0

    def solicitar_recarga_cloud(self):
        """Simula el Lead Time de la recarga de créditos."""
        self.recarga_pendiente = True
        tiempo_entrega = random.expovariate(1 / self.lead_time)
        yield self.env.timeout(tiempo_entrega)
        yield self.creditos.put(self.cantidad_recarga)
        self.recarga_pendiente = False

    def controlar_inventario(self):
        """Política continua (s, Q)."""
        while True:
            if self.creditos.level <= self.punto_reorden and not self.recarga_pendiente:
                self.env.process(self.solicitar_recarga_cloud())
            yield self.env.timeout(0.1)


In [5]:
def peticion_api(env, api):
    llegada = env.now

    with api.gpus.request() as req:
        yield req

        # Tiempo de espera en cola (Wq)
        api.tiempos_espera.append(env.now - llegada)

        # Tiempo de servicio
        tiempo_servicio = random.expovariate(MU)
        yield env.timeout(tiempo_servicio)

        # Consumo de crédito
        if api.creditos.level > 0:
            yield api.creditos.get(1)
        else:
            api.predicciones_fallidas += 1


def generador_peticiones(env, api):
    while True:
        yield env.timeout(random.expovariate(LAMBDA))
        env.process(peticion_api(env, api))

In [6]:
def calcular_intervalo_confianza(datos, confianza=0.95):
    n = len(datos)
    media = np.mean(datos)
    error = st.sem(datos)
    h = error * st.t.ppf((1 + confianza) / 2., n-1)
    return media, media - h, media + h

## 3. Ejecución y Visualización Inicial
Correremos el modelo una vez para observar la caída del inventario ("Dientes de sierra") y analizar cómo interactúan las colas y los reabastecimientos.


In [7]:
def ejecutar_experimento():
    resultados_wq = []
    resultados_fallidas = []

    for _ in range(REPLICAS):
        env = simpy.Environment()
        api = API_MachineLearning(env)

        env.process(generador_peticiones(env, api))
        env.process(api.controlar_inventario())
        env.run(until=TIEMPO_SIMULACION)

        resultados_wq.append(np.mean(api.tiempos_espera))
        resultados_fallidas.append(api.predicciones_fallidas)

    media_wq, li, ls = calcular_intervalo_confianza(resultados_wq)

    print("===== RESULTADOS =====")
    print(f"Intervalo de Confianza 95% Wq: [{li:.4f}, {ls:.4f}] minutos")
    print(f"Promedio de predicciones fallidas: {np.mean(resultados_fallidas):.2f}")

    rho = LAMBDA / (SERVIDORES * MU)
    print(f"Utilización teórica GPUs (ρ): {rho:.2f}")

In [8]:
ejecutar_experimento()

===== RESULTADOS =====
Intervalo de Confianza 95% Wq: [0.0000, 0.0000] minutos
Promedio de predicciones fallidas: 82.37
Utilización teórica GPUs (ρ): 0.12
